# 20 — Tolerance Interval vs. Sample Size / 容差区间与样本大小

**Chapter 20 — File 2 of 2**

## Summary / 摘要

The width of a tolerance interval decreases as sample size increases, following an inverse square root relationship. With larger samples, the estimation uncertainty diminishes, resulting in tighter bounds. This notebook demonstrates how the tolerance interval width changes when varying sample sizes from 5 to 14 observations, showing the practical trade-off between data collection cost and desired precision.

容差区间的宽度随着样本大小的增加而减小，遵循倒平方根关系。样本越大，估计不确定性越小，导致边界越紧。此笔记本演示了当样本大小从5到14个观察值变化时，容差区间宽度如何变化，显示了数据收集成本和所需精度之间的实际权衡。

## Step 1 — Import Libraries / 导入库

In [ ]:
# Import required libraries
# 导入所需库
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

# Set random seed for reproducibility
# 设置随机种子以保证可重复性
np.random.seed(42)

## Step 2 — Define Parameters and Sample Sizes / 定义参数和样本大小

In [ ]:
# Population parameters
# 总体参数
mu = 100
sigma = 15

# Tolerance interval parameters
# 容差区间参数
coverage = 0.95  # 95% coverage / 95%覆盖
confidence = 0.99  # 99% confidence / 99%置信
alpha = 1 - confidence

# Critical values (same for all sample sizes)
# 临界值（对所有样本大小相同）
z_coverage = stats.norm.ppf((1 + coverage) / 2)

# Range of sample sizes to test
# 要测试的样本大小范围
sample_sizes = np.arange(5, 15)  # n=5 to n=14 / n=5到n=14

print(f"Population mean: {mu}")
print(f"Population std: {sigma}")
print(f"Coverage: {coverage*100:.0f}%, Confidence: {confidence*100:.0f}%")
print(f"Sample sizes to evaluate: {sample_sizes}")

## Step 3 — Calculate Tolerance Intervals for Different Sample Sizes / 为不同样本大小计算容差区间

In [ ]:
# Store results
# 存储结果
interval_widths = []
lower_bounds = []
upper_bounds = []

# Calculate tolerance interval for each sample size
# 为每个样本大小计算容差区间
for n in sample_sizes:
    # Generate sample data
    # 生成样本数据
    sample_data = np.random.normal(mu, sigma, n)
    
    # Calculate sample statistics
    # 计算样本统计量
    sample_mean = np.mean(sample_data)
    sample_std = np.std(sample_data, ddof=1)
    
    # Get chi-square critical value
    # 获取卡方临界值
    chi2_val = stats.chi2.ppf(1 - alpha, df=n-1)
    
    # Calculate margin of error
    # 计算误差范围
    margin = z_coverage * sigma * np.sqrt(1 + 1/n)  # Use population sigma for consistency
                                                      # 使用总体标准差以保持一致性
    
    # Calculate bounds
    # 计算边界
    lower = mu - margin
    upper = mu + margin
    width = upper - lower
    
    # Store results
    # 存储结果
    interval_widths.append(width)
    lower_bounds.append(lower)
    upper_bounds.append(upper)
    
    print(f"n={n:2d}: Width={width:.2f}, Bounds=[{lower:.2f}, {upper:.2f}]")

In [ ]:
# Convert to arrays for plotting
# 转换为数组以进行绘制
interval_widths = np.array(interval_widths)
lower_bounds = np.array(lower_bounds)
upper_bounds = np.array(upper_bounds)

## Step 4 — Plot Tolerance Interval Width vs. Sample Size / 绘制容差区间宽度对样本大小

In [ ]:
# Create comprehensive visualization
# 创建综合可视化
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Tolerance interval width vs. sample size
# 图1: 容差区间宽度对样本大小
axes[0, 0].plot(sample_sizes, interval_widths, marker='o', linewidth=2, markersize=8, color='blue')
axes[0, 0].set_xlabel('Sample Size (n)')
axes[0, 0].set_ylabel('Interval Width')
axes[0, 0].set_title('Tolerance Interval Width vs. Sample Size')
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].set_xticks(sample_sizes)

# Add annotation for trend
# 为趋势添加注释
axes[0, 0].annotate('Decreasing trend\n(Inverse sqrt relationship)', 
                    xy=(sample_sizes[-1], interval_widths[-1]), 
                    xytext=(sample_sizes[-3], interval_widths[-3] + 1),
                    arrowprops=dict(arrowstyle='->', color='red'),
                    fontsize=10)

# Plot 2: Log scale to show inverse sqrt relationship
# 图2: 对数刻度显示倒平方根关系
axes[0, 1].loglog(sample_sizes, interval_widths, marker='s', linewidth=2, markersize=8, color='green')
axes[0, 1].set_xlabel('Sample Size (n) - log scale')
axes[0, 1].set_ylabel('Interval Width - log scale')
axes[0, 1].set_title('Log-Log Plot: Inverse Sqrt Relationship')
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: Tolerance intervals as error bars
# 图3: 容差区间作为误差条
axes[1, 0].errorbar(sample_sizes, np.ones_like(sample_sizes) * mu, 
                    yerr=interval_widths/2,
                    fmt='o', linewidth=2, markersize=8, capsize=5, color='purple')
axes[1, 0].axhline(mu, color='black', linestyle='-', linewidth=1)
axes[1, 0].set_xlabel('Sample Size (n)')
axes[1, 0].set_ylabel('Center (mean)')
axes[1, 0].set_title('Tolerance Intervals at Different Sample Sizes')
axes[1, 0].set_xticks(sample_sizes)
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].set_ylim([95, 105])

# Plot 4: Rate of change in width
# 图4: 宽度变化率
width_reduction = (interval_widths[0] - interval_widths) / interval_widths[0] * 100
axes[1, 1].bar(sample_sizes, width_reduction, color='orange', edgecolor='black', alpha=0.7)
axes[1, 1].set_xlabel('Sample Size (n)')
axes[1, 1].set_ylabel('Width Reduction (%)')
axes[1, 1].set_title('Cumulative Width Reduction from n=5')
axes[1, 1].set_xticks(sample_sizes)
axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# Summary statistics
# 摘要统计
print(f"\nTolerance Interval Width Summary:")
print(f"At n=5:  Width = {interval_widths[0]:.2f}")
print(f"At n=14: Width = {interval_widths[-1]:.2f}")
print(f"Reduction: {interval_widths[0] - interval_widths[-1]:.2f} ({(1 - interval_widths[-1]/interval_widths[0])*100:.1f}% decrease)")
print(f"\nWidth reduction is proportional to 1/sqrt(n):")
print(f"sqrt(5) = {np.sqrt(5):.2f}, sqrt(14) = {np.sqrt(14):.2f}")
print(f"Ratio: {np.sqrt(5)/np.sqrt(14):.2f} ≈ {interval_widths[0]/interval_widths[-1]:.2f}")

In [ ]:
# Create a table of values
# 创建值表
print(f"\nDetailed Results Table:")
print(f"{'n':>3} | {'Width':>8} | {'Lower':>8} | {'Upper':>8} | {'% Reduction':>12}")
print("-" * 50)
for i, n in enumerate(sample_sizes):
    reduction_pct = (1 - interval_widths[i]/interval_widths[0]) * 100
    print(f"{n:3d} | {interval_widths[i]:8.2f} | {lower_bounds[i]:8.2f} | {upper_bounds[i]:8.2f} | {reduction_pct:11.1f}%")

## Learning Notes / 学习笔记

- **Statistical Concept**: Tolerance interval width decreases with sample size following the relationship: width ∝ 1/√n. This inverse square root law means quadrupling the sample size only halves the interval width. The relationship comes from the standard error term √(1+1/n) in the margin calculation, demonstrating fundamental sampling variability principles.
  
  **统计概念**: 容差区间宽度随样本大小减小，遵循以下关系：宽度∝1/√n。这个倒平方根定律意味着将样本大小增加四倍只能将区间宽度减半。该关系来自裕度计算中的标准误差项√(1+1/n)，演示了基本的采样变异性原则。

- **ML Application**: Sample size planning is critical in production systems where tolerance intervals define operational specifications. Understanding the width-sample size tradeoff helps optimize data collection budgets in quality control, sensor calibration, and process validation. Larger samples provide tighter specifications but at higher cost, requiring careful cost-benefit analysis.
  
  **ML应用**: 样本大小规划在容差区间定义操作规格的生产系统中至关重要。理解宽度-样本大小权衡有助于优化质量控制、传感器校准和流程验证中的数据收集预算。较大的样本提供更紧的规格，但成本更高，需要仔细的成本效益分析。

➡️ **Next**: `../chapter_21/01_confidence_interval_50.ipynb`

## Complete Code / 完整代码一览

In [ ]:
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

np.random.seed(42)
mu, sigma = 100, 15
coverage, confidence = 0.95, 0.99
alpha = 1 - confidence
z_coverage = stats.norm.ppf((1 + coverage) / 2)

sample_sizes = np.arange(5, 15)
interval_widths = []

for n in sample_sizes:
    margin = z_coverage * sigma * np.sqrt(1 + 1/n)
    width = 2 * margin
    interval_widths.append(width)
    print(f"n={n}: Width={width:.2f}")

plt.plot(sample_sizes, interval_widths, marker='o', linewidth=2)
plt.xlabel('Sample Size (n)')
plt.ylabel('Interval Width')
plt.title('Tolerance Interval Width vs. Sample Size')
plt.grid(True, alpha=0.3)
plt.show()